# QuantUI-local — Quantum Chemistry Calculator

Run PySCF quantum chemistry calculations directly in this notebook.

**How to use:**
1. Select or enter a molecule in **Molecule Input**
2. Choose a method and basis set in **Calculation Setup**
3. Click **Run Calculation** — results appear below
4. Optionally add results to **Compare** or **Export** a standalone script

> **Platform note:** PySCF requires Linux, macOS, or WSL.  \
> Windows users: `apptainer run quantui-local.sif`


In [ ]:
# Environment check — verifies correct conda environment.
# Tagged skip-execution and remove-input so it is hidden in Voilà.
import sys as _sys
_env = _sys.prefix
if "quantui" not in _env.lower():
    print(f"Warning: active environment may not be quantui-local")
    print(f"Active: {_env}")
    print("Run: conda activate quantui-local")


In [ ]:
import threading
import ipywidgets as widgets
from IPython.display import display, HTML

import quantui
from quantui import (
    Molecule, parse_xyz_input,
    MOLECULE_LIBRARY, SUPPORTED_METHODS, SUPPORTED_BASIS_SETS,
    DEFAULT_METHOD, DEFAULT_BASIS, DEFAULT_CHARGE, DEFAULT_MULTIPLICITY,
    session_can_handle, get_session_resources,
    PUBCHEM_AVAILABLE, VISUALIZATION_AVAILABLE, ASE_AVAILABLE,
    QUICK_START_TEMPLATES,
)

# Optional — degrade gracefully if unavailable
try:
    from quantui import run_in_session, SessionResult
    PYSCF_AVAILABLE = True
except (ImportError, AttributeError):
    PYSCF_AVAILABLE = False

try:
    from quantui import student_friendly_fetch
except (ImportError, AttributeError):
    student_friendly_fetch = None

try:
    from quantui import display_molecule
except (ImportError, AttributeError):
    display_molecule = None

try:
    from quantui import preoptimize
    PREOPT_AVAILABLE = True
except (ImportError, AttributeError):
    PREOPT_AVAILABLE = False

# Mutable session state shared across all callbacks
_state = {"molecule": None, "last_result": None, "results": []}

# ── Global app styles ─────────────────────────────────────────────────────────
# Permanent — not toggled by dark mode (dark mode uses filter inversion on top).
display(HTML("""<style>
/* System font stack ---------------------------------------------------- */
body, p, span, li, td, th, label, input, select, textarea, blockquote,
.jp-OutputArea-output, .widget-html-content, .widget-label-basic,
.widget-label {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", system-ui,
                 Roboto, "Helvetica Neue", Arial, sans-serif !important;
    -webkit-font-smoothing: antialiased;
}

/* App title (h1 in the markdown cell) ---------------------------------- */
h1 {
    font-size: 20px !important;
    font-weight: 700 !important;
    color: #1e293b !important;
    letter-spacing: -0.01em !important;
    margin: 10px 0 4px !important;
    border-bottom: none !important;
}

/* Section headers ------------------------------------------------------- */
/* Overrides the inline h3 styles used in every display cell. */
h3 {
    font-size: 11px !important;
    font-weight: 700 !important;
    letter-spacing: 0.09em !important;
    text-transform: uppercase !important;
    color: #64748b !important;
    margin: 24px 0 10px !important;
    padding-bottom: 5px !important;
    border-bottom: 1px solid #e2e8f0 !important;
}

/* Rounded corners on inputs, dropdowns, and buttons -------------------- */
.widget-text input, .widget-textarea textarea {
    border-color: #d1d5db !important;
    border-radius: 5px !important;
}
.widget-dropdown select { border-radius: 5px !important; }
.widget-button, .widget-toggle-button { border-radius: 5px !important; }
</style>"""))

# ── Dark mode toggle ─────────────────────────────────────────────────────────
# Uses CSS filter invert+hue-rotate on the html element so it works with all
# inline-styled elements. canvas/img/iframe are re-inverted to keep their
# original appearance (e.g. the py3Dmol 3D viewer).
# Map theme name → hue-rotate angle.  Light uses no filter.
_THEME_HUE = {"Dark": 180, "Dark Blue": 200, "Dark Maroon": 160}


def _theme_css(theme: str) -> str:
    """Return the CSS filter block for *theme*, or '' for Light."""
    if theme not in _THEME_HUE:
        return ""
    deg = _THEME_HUE[theme]
    return (
        "<style>"
        f"html {{ filter: invert(1) hue-rotate({deg}deg) !important; }}"
        "canvas, img, iframe, video "
        f"{{ filter: invert(1) hue-rotate({deg}deg) !important; }}"
        "</style>"
    )


_theme_style = widgets.Output(
    layout=widgets.Layout(height="0px", overflow="hidden", margin="0", padding="0")
)

theme_btn = widgets.ToggleButtons(
    options=["Light", "Dark", "Dark Blue", "Dark Maroon"],
    value="Dark",
    description="Theme:",
    style={
        "description_width": "48px",
        "button_width": "90px",
    },
    layout=widgets.Layout(margin="0"),
)


def _toggle_theme(change):
    _theme_style.clear_output()
    css = _theme_css(change["new"])
    if css:
        with _theme_style:
            display(HTML(css))


theme_btn.observe(_toggle_theme, names="value")

# Apply Dark theme on startup
with _theme_style:
    display(HTML(_theme_css("Dark")))

display(widgets.HBox(
    [theme_btn],
    layout=widgets.Layout(justify_content="flex-end", margin="0 0 4px"),
))
display(_theme_style)

# ── Status panel ────────────────────────────────────────────────────────────
_cores, _mem_gb = get_session_resources()
_mem = f"{_mem_gb} GB" if _mem_gb is not None else "unknown"


def _ok(flag, extra=""):
    tick = '<span style="color:#22c55e">&#10003;</span>'
    cross = '<span style="color:#ef4444">&#10007;</span>'
    return (tick if flag else cross) + (" " + extra if extra else "")


_items = [
    ("PySCF (calculations)", _ok(PYSCF_AVAILABLE,
        "" if PYSCF_AVAILABLE else "&mdash; Linux / macOS / WSL required")),
    ("ASE (structure I/O, opt.)", _ok(ASE_AVAILABLE)),
    ("PubChem search", _ok(PUBCHEM_AVAILABLE)),
    ("3D viewer (py3Dmol)", _ok(VISUALIZATION_AVAILABLE)),
    ("CPU cores / Memory", f"<b>{_cores}</b> cores / <b>{_mem}</b>"),
]
_rows = "".join(
    f'<tr><td style="padding:3px 16px 3px 0;color:#64748b;font-size:13px">{k}</td>'
    f'<td style="font-size:13px">{v}</td></tr>'
    for k, v in _items
)
display(HTML(
    f'<div style="background:#f8fafc;border:1px solid #e2e8f0;border-left:4px solid #3b82f6;'
    f'padding:12px 16px;border-radius:6px;margin:4px 0 8px">'
    f'<span style="font-weight:600;font-size:14px;color:#1e293b">'
    f"QuantUI-local {quantui.__version__}</span>"
    f'<table style="margin-top:8px;border-collapse:collapse">{_rows}</table>'
    f"</div>"
))


In [ ]:
import io
import re
import time

from quantui.progress import StepProgress
import quantui.calc_log as _calc_log

# ── Shared output widgets ────────────────────────────────────────────────────
mol_info_html = widgets.HTML(
    value='<i style="color:#888">No molecule loaded yet.</i>'
)
mol_summary_compact = widgets.HTML(value="")
viz_output    = widgets.Output(layout=widgets.Layout(min_height="50px"))
run_output    = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #c0ccd8", min_height="80px", max_height="400px",
        padding="8px", overflow_y="auto",
    )
)
with run_output:
    display(HTML(
        '<p style="color:#999;font-style:italic;font-size:13px;margin:2px 0">'
        "No calculation run yet. PySCF output and any errors will appear here."
        "</p>"
    ))
result_output     = widgets.Output()
comparison_output = widgets.Output()
notes_output      = widgets.Output()
perf_estimate_html = widgets.HTML()

# ── Step indicator ────────────────────────────────────────────────────────────
step_progress = StepProgress(["Choose molecule", "Set method", "Run", "Results"])
step_progress.start(0)

# ── Calculation setup (defined here so _set_molecule can update them) ────────
method_dd = widgets.Dropdown(
    options=SUPPORTED_METHODS, value=DEFAULT_METHOD,
    description="Method:", style={"description_width": "100px"},
    layout=widgets.Layout(width="260px"),
)
basis_dd = widgets.Dropdown(
    options=SUPPORTED_BASIS_SETS, value=DEFAULT_BASIS,
    description="Basis Set:", style={"description_width": "100px"},
    layout=widgets.Layout(width="260px"),
)
charge_si = widgets.BoundedIntText(
    value=DEFAULT_CHARGE, min=-10, max=10,
    description="Charge:", style={"description_width": "100px"},
    layout=widgets.Layout(width="190px"),
)
mult_si = widgets.BoundedIntText(
    value=DEFAULT_MULTIPLICITY, min=1, max=10,
    description="Multiplicity:", style={"description_width": "100px"},
    layout=widgets.Layout(width="190px"),
)
preopt_cb = widgets.Checkbox(
    value=False,
    description="Pre-optimize geometry (fast LJ force-field)",
    disabled=not PREOPT_AVAILABLE,
    layout=widgets.Layout(width="400px"),
)

# ── Calculation type + extra options ──────────────────────────────────────────
calc_type_dd = widgets.Dropdown(
    options=["Single Point", "Geometry Opt", "Frequency", "UV-Vis (TD-DFT)"],
    value="Single Point",
    description="Calc. Type:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="310px"),
)
fmax_fi = widgets.BoundedFloatText(
    value=0.05, min=0.001, max=1.0, step=0.005,
    description="Force thr. (eV/Å):",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="270px"),
)
max_steps_si = widgets.BoundedIntText(
    value=200, min=10, max=1000,
    description="Max steps:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="200px"),
)
nstates_si = widgets.BoundedIntText(
    value=10, min=1, max=50,
    description="# states:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="180px"),
)
calc_extra_opts = widgets.VBox([])


def _on_calc_type_change(change):
    ct = change["new"]
    if ct == "Geometry Opt":
        calc_extra_opts.children = [
            widgets.HBox(
                [fmax_fi, max_steps_si],
                layout=widgets.Layout(gap="8px"),
            ),
        ]
    elif ct == "UV-Vis (TD-DFT)":
        calc_extra_opts.children = [
            nstates_si,
            widgets.HTML(
                '<span style="color:#b45309;font-size:12px">⚠ Requires a DFT '
                "functional (e.g. B3LYP, PBE0). RHF/UHF will run TDHF (CIS) "
                "instead.</span>"
            ),
        ]
    else:
        calc_extra_opts.children = []


calc_type_dd.observe(_on_calc_type_change, names="value")

# ── Context-help buttons (next to Method and Basis dropdowns) ────────────────
method_help_btn = widgets.Button(
    description="?", button_style="",
    layout=widgets.Layout(width="28px", height="28px"),
    tooltip="RHF vs UHF — opens Help tab",
)
basis_help_btn = widgets.Button(
    description="?", button_style="",
    layout=widgets.Layout(width="28px", height="28px"),
    tooltip="Choosing a basis set — opens Help tab",
)

# ── Run widgets ──────────────────────────────────────────────────────────────
run_btn = widgets.Button(
    description="Run Calculation", button_style="success", icon="play",
    disabled=True, layout=widgets.Layout(width="200px", height="36px"),
)
run_status = widgets.Label()

# ── Log clear button ─────────────────────────────────────────────────────────
log_clear_btn = widgets.Button(
    description="Clear", button_style="", icon="times",
    layout=widgets.Layout(width="90px", height="26px"),
    tooltip="Clear calculation output",
)

def _clear_log(btn):
    run_output.clear_output()

log_clear_btn.on_click(_clear_log)

# ── Comparison / export widgets ───────────────────────────────────────────────
accumulate_btn = widgets.Button(
    description="Add to Comparison", button_style="info", icon="plus",
    disabled=True, layout=widgets.Layout(width="190px"),
)
clear_btn = widgets.Button(
    description="Clear", button_style="warning", icon="trash",
    layout=widgets.Layout(width="100px"),
)
export_btn = widgets.Button(
    description="Export Script", button_style="", icon="download",
    disabled=True, layout=widgets.Layout(width="160px"),
)
export_status = widgets.Label()


# ── Thread-safe log capture ───────────────────────────────────────────────────
_RE_CYCLE = re.compile(
    r"cycle=\s*(\d+)\s+E=\s*([\-\d\.]+)\s+delta_E=\s*([\-\d\.Ee+\-]+)"
)
_RE_CONV = re.compile(r"converged SCF energy\s*=\s*([\-\d\.]+)")


class _LogCapture:
    '''Write PySCF output to an Output widget and capture it to a buffer.'''
    def __init__(self, output_widget):
        self._w = output_widget
        self._buf = io.StringIO()
        self._line_buf = ""

    def write(self, text: str) -> None:
        if not text:
            return
        self._w.append_stdout(text)
        self._buf.write(text)
        # Scan complete lines for SCF progress and update the status label.
        self._line_buf += text
        while "\n" in self._line_buf:
            line, self._line_buf = self._line_buf.split("\n", 1)
            m = _RE_CYCLE.search(line)
            if m:
                n, delta = m.group(1), m.group(3)
                try:
                    run_status.value = f"SCF cycle {n}  ·  ΔE = {float(delta):.4g} Ha"
                except Exception:
                    run_status.value = f"SCF cycle {n}"
                continue
            m = _RE_CONV.search(line)
            if m:
                run_status.value = "SCF converged ✓"

    def flush(self) -> None:
        pass

    def getvalue(self) -> str:
        return self._buf.getvalue()


# Placeholders — overwritten by later cells once they execute.
_refresh_results_browser = lambda: None  # noqa: E731
_show_help_topic = lambda topic: None  # noqa: E731
_populate_compare_list = lambda: None  # noqa: E731


# ── Callbacks ─────────────────────────────────────────────────────────────────

def _set_molecule(mol, label=""):
    '''Update shared state and refresh dependent widgets.'''
    _state["molecule"] = mol
    run_btn.disabled    = False
    export_btn.disabled = False

    try:
        n_e = mol.get_electron_count()
        e_str = f"{n_e} electrons"
    except Exception:
        e_str = ""

    _lbl = f'<br><small style="color:#777">{label}</small>' if label else ""
    _summary = (
        f'<b style="font-size:15px">{mol.get_formula()}</b>'
        f'&ensp;<span style="color:#555;font-size:13px">'
        f"{len(mol.atoms)} atoms"
        + (f" &bull; {e_str}" if e_str else "")
        + f" &bull; charge {mol.charge} &bull; mult {mol.multiplicity}"
        + f"</span>{_lbl}"
    )
    mol_info_html.value = _summary
    mol_summary_compact.value = (
        f'<div style="background:#f0f9ff;border:1px solid #bae6fd;'
        f'border-radius:6px;padding:7px 14px;font-size:14px;display:inline-block">'
        f"{_summary}</div>"
    )

    charge_si.value = mol.charge
    mult_si.value   = mol.multiplicity
    if mol.multiplicity > 1 and method_dd.value == "RHF":
        method_dd.value = "UHF"

    viz_output.clear_output(wait=True)
    if display_molecule is not None:
        with viz_output:
            display_molecule(mol)

    _update_notes()

    # Advance step indicator — but not during a running calculation (e.g. preopt)
    if step_progress._states[2] != "active":
        if step_progress._states[2] in ("done", "fail"):
            # Fresh molecule after a completed run — reset indicator
            step_progress.reset()
        step_progress.complete(0)
        step_progress.start(1)

    # Update time estimate for the newly loaded molecule
    _update_estimate()

    # Collapse the molecule input panel to the compact summary + 3D viewer
    mol_input_container.children = [mol_input_collapsed, viz_output]


def _format_result(r):
    _conv   = "Yes" if r.converged else "No (treat results with caution)"
    _cc     = "green" if r.converged else "#c00"
    _gap    = (
        f"{r.homo_lumo_gap_ev:.4f} eV"
        if r.homo_lumo_gap_ev is not None else "N/A"
    )
    _rows = "".join(
        f'<tr>'
        f'<td style="padding:3px 18px 3px 0;color:#444">{k}</td>'
        f'<td style="color:{vc}">{v}</td>'
        f"</tr>"
        for k, v, vc in [
            ("Total energy",
             f"{r.energy_hartree:.8f} Ha &ensp;({r.energy_ev:.4f} eV)", "#000"),
            ("HOMO-LUMO gap",   _gap, "#000"),
            ("SCF converged",   _conv, _cc),
            ("SCF iterations",  str(r.n_iterations), "#000"),
        ]
    )
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f"<b>{r.formula} &mdash; {r.method}/{r.basis}</b>"
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _format_opt_result(r):
    '''Format an OptimizationResult as an HTML result card.'''
    _conv = "Yes" if r.converged else "No (max steps reached)"
    _cc = "green" if r.converged else "#c00"
    _rows = "".join(
        f'<tr>'
        f'<td style="padding:3px 18px 3px 0;color:#444">{k}</td>'
        f'<td style="color:{vc}">{v}</td>'
        f"</tr>"
        for k, v, vc in [
            ("Final energy",  f"{r.energy_hartree:.8f} Ha", "#000"),
            ("Energy change", f"{r.energy_change_hartree:+.6f} Ha", "#000"),
            ("Opt converged", _conv, _cc),
            ("Steps taken",   str(r.n_steps), "#000"),
            ("Geometry RMSD", f"{r.rmsd_angstrom:.4f} Å", "#000"),
        ]
    )
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f"<b>Geometry Optimisation &mdash; {r.formula} ({r.method}/{r.basis})</b>"
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _format_freq_result(r):
    '''Format a FreqResult as an HTML result card.'''
    _conv = "Yes" if r.converged else "No (treat with caution)"
    _cc = "green" if r.converged else "#c00"
    n_real = r.n_real_modes()
    n_imag = r.n_imaginary_modes()
    real_freqs = sorted(f for f in r.frequencies_cm1 if f > 0)[:6]
    freq_str = "  ".join(f"{f:.1f}" for f in real_freqs)
    if len([f for f in r.frequencies_cm1 if f > 0]) > 6:
        freq_str += " …"
    imag_note = ""
    if n_imag > 0:
        imag_note = (
            f'<tr><td style="padding:3px 18px 3px 0;color:#444">Imaginary modes</td>'
            f'<td style="color:#c00">{n_imag} — geometry may not be a minimum</td></tr>'
        )
    _rows = (
        f'<tr><td style="padding:3px 18px 3px 0;color:#444">SCF energy</td>'
        f'<td style="color:#000">{r.energy_hartree:.8f} Ha</td></tr>'
        f'<tr><td style="padding:3px 18px 3px 0;color:#444">SCF converged</td>'
        f'<td style="color:{_cc}">{_conv}</td></tr>'
        f'<tr><td style="padding:3px 18px 3px 0;color:#444">Real modes</td>'
        f'<td style="color:#000">{n_real}</td></tr>'
        + imag_note
        + (f'<tr><td style="padding:3px 18px 3px 0;color:#444">Frequencies (cm⁻¹)</td>'
           f'<td style="color:#000;font-family:monospace">{freq_str or "none"}</td></tr>'
           if real_freqs else "")
        + f'<tr><td style="padding:3px 18px 3px 0;color:#444">ZPVE</td>'
        f'<td style="color:#000">{r.zpve_hartree:.6f} Ha '
        f'({r.zpve_hartree * 27.211386245988:.4f} eV)</td></tr>'
    )
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f"<b>Frequency Analysis &mdash; {r.formula} ({r.method}/{r.basis})</b>"
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _format_tddft_result(r):
    '''Format a TDDFTResult as an HTML result card with excitation table.'''
    _conv = "Yes" if r.converged else "No (treat with caution)"
    _cc = "green" if r.converged else "#c00"
    header_rows = (
        f'<tr><td style="padding:3px 18px 3px 0;color:#444">Ground-state energy</td>'
        f'<td style="color:#000">{r.energy_hartree:.8f} Ha</td></tr>'
        f'<tr><td style="padding:3px 18px 3px 0;color:#444">SCF converged</td>'
        f'<td style="color:{_cc}">{_conv}</td></tr>'
        f'<tr><td style="padding:3px 18px 3px 0;color:#444">States computed</td>'
        f'<td style="color:#000">{len(r.excitation_energies_ev)}</td></tr>'
    )
    exc_table = ""
    if r.excitation_energies_ev:
        wl = r.wavelengths_nm()
        exc_rows = []
        for i, (e_ev, f_osc) in enumerate(
            zip(r.excitation_energies_ev[:8], r.oscillator_strengths[:8]), 1
        ):
            bold = "font-weight:bold" if f_osc > 0.05 else ""
            exc_rows.append(
                f'<tr style="{bold}">'
                f'<td style="padding:2px 12px 2px 0;color:#555">S{i}</td>'
                f'<td style="padding:2px 12px 2px 0;color:#000">{e_ev:.3f} eV</td>'
                f'<td style="padding:2px 12px 2px 0;color:#000">{wl[i - 1]:.1f} nm</td>'
                f'<td style="padding:2px 4px 2px 0;color:#000">f = {f_osc:.4f}</td>'
                f"</tr>"
            )
        if len(r.excitation_energies_ev) > 8:
            exc_rows.append(
                f'<tr><td colspan="4" style="color:#888;font-size:12px">… '
                f"and {len(r.excitation_energies_ev) - 8} more states</td></tr>"
            )
        exc_table = (
            '<tr><td colspan="2" style="padding:8px 0 2px;color:#444;font-weight:bold">'
            "Vertical excitations:</td></tr>"
            '<tr>'
            '<th style="text-align:left;color:#555;font-size:12px;padding:2px 12px 2px 0">State</th>'
            '<th style="text-align:left;color:#555;font-size:12px;padding:2px 12px 2px 0">Energy</th>'
            '<th style="text-align:left;color:#555;font-size:12px;padding:2px 12px 2px 0">λ</th>'
            '<th style="text-align:left;color:#555;font-size:12px">Osc. str.</th></tr>'
            + "".join(exc_rows)
        )
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f"<b>TD-DFT / UV-Vis &mdash; {r.formula} ({r.method}/{r.basis})</b>"
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{header_rows}{exc_table}</table></div>"
    )


def _do_run(btn):
    mol = _state["molecule"]
    if mol is None:
        run_status.value = "Load a molecule first."
        return
    run_btn.disabled = True
    run_status.value = "Starting..."
    run_output.clear_output()
    result_output.clear_output()

    # Advance step indicator: method confirmed → running
    step_progress.complete(1)
    step_progress.start(2)

    _calc_log.log_event(
        "calc_start",
        f"{method_dd.value}/{basis_dd.value} on {mol.get_formula()}",
        n_atoms=len(mol.atoms),
    )
    _run_wall_t = time.perf_counter()

    def _thread():
        log = _LogCapture(run_output)
        try:
            calc_mol = mol
            if preopt_cb.value and PREOPT_AVAILABLE:
                run_status.value = "Pre-optimizing..."
                calc_mol, _rmsd = preoptimize(mol)
                _set_molecule(calc_mol, f"Geometry pre-optimized (LJ, RMSD={_rmsd:.3f} Å)")

            # ── Dispatch to the right backend based on Calc. Type ─────────────
            ct = calc_type_dd.value
            if ct == "Geometry Opt":
                run_status.value = "Optimizing geometry..."
                from quantui import optimize_geometry
                result = optimize_geometry(
                    molecule=calc_mol,
                    method=method_dd.value,
                    basis=basis_dd.value,
                    fmax=fmax_fi.value,
                    steps=max_steps_si.value,
                    progress_stream=log,
                )
                result_html = _format_opt_result(result)
                save_spectra, save_type = {}, "geometry_opt"
            elif ct == "Frequency":
                run_status.value = "Computing frequencies (SCF + Hessian)..."
                from quantui.freq_calc import run_freq_calc
                result = run_freq_calc(
                    molecule=calc_mol,
                    method=method_dd.value,
                    basis=basis_dd.value,
                    progress_stream=log,
                )
                result_html = _format_freq_result(result)
                save_spectra = {"ir": {
                    "frequencies_cm1": result.frequencies_cm1,
                    "ir_intensities": result.ir_intensities,
                    "zpve_hartree": result.zpve_hartree,
                }}
                save_type = "frequency"
            elif ct == "UV-Vis (TD-DFT)":
                run_status.value = "Running TD-DFT excited states..."
                from quantui.tddft_calc import run_tddft_calc
                result = run_tddft_calc(
                    molecule=calc_mol,
                    method=method_dd.value,
                    basis=basis_dd.value,
                    nstates=nstates_si.value,
                    progress_stream=log,
                )
                result_html = _format_tddft_result(result)
                save_spectra = {"uv_vis": {
                    "excitation_energies_ev": result.excitation_energies_ev,
                    "oscillator_strengths": result.oscillator_strengths,
                    "wavelengths_nm": result.wavelengths_nm(),
                }}
                save_type = "tddft"
            else:  # Single Point
                run_status.value = "Calculating..."
                result = run_in_session(
                    molecule=calc_mol,
                    method=method_dd.value,
                    basis=basis_dd.value,
                    progress_stream=log,
                )
                result_html = _format_result(result)
                save_spectra, save_type = {}, "single_point"
            _elapsed = time.perf_counter() - _run_wall_t

            _state["last_result"] = result
            accumulate_btn.disabled = False

            result_output.append_display_data(HTML(result_html))
            run_status.value = f"Done in {_elapsed:.1f} s."

            # Advance step indicator: run complete → results ready
            step_progress.complete(2)
            step_progress.complete(3)

            # Persist result to disk
            try:
                from quantui import save_result
                save_result(
                    result, pyscf_log=log.getvalue(),
                    calc_type=save_type, spectra=save_spectra,
                )
                _refresh_results_browser()
                _populate_compare_list()
            except Exception:
                pass

            # Log performance record and refresh the estimate widget
            try:
                _calc_log.log_calculation(
                    formula=result.formula,
                    n_atoms=len(calc_mol.atoms),
                    n_electrons=calc_mol.get_electron_count(),
                    method=result.method,
                    basis=result.basis,
                    n_iterations=getattr(result, "n_iterations", -1),
                    elapsed_s=_elapsed,
                    converged=result.converged,
                )
                _calc_log.log_event(
                    "calc_done",
                    f"{result.method}/{result.basis} on {result.formula}",
                    elapsed_s=round(_elapsed, 2),
                    converged=result.converged,
                )
                _update_estimate()
            except Exception:
                pass

        except ImportError:
            log.write(
                "PySCF is not available in this environment.\n"
                "PySCF requires Linux, macOS, or WSL.\n"
                "On Windows: use the Apptainer container.\n"
                "  apptainer run quantui-local.sif\n"
            )
            run_status.value = "PySCF unavailable."
            step_progress.fail(2, "PySCF unavailable")
            _calc_log.log_event("calc_error", "PySCF unavailable")

        except Exception as exc:
            import traceback
            _elapsed = time.perf_counter() - _run_wall_t
            log.write(f"Error: {exc}\n\n{traceback.format_exc()}")
            run_status.value = "Error — see Calculation Output below."
            step_progress.fail(2, str(exc)[:60])
            _calc_log.log_event("calc_error", str(exc)[:200], elapsed_s=round(_elapsed, 2))

        finally:
            run_btn.disabled = False

    threading.Thread(target=_thread, daemon=True).start()

run_btn.on_click(_do_run)


def _update_notes(change=None):
    notes_output.clear_output(wait=True)
    mol = _state["molecule"]
    if mol is None:
        return
    try:
        from quantui import PySCFCalculation
        calc  = PySCFCalculation(mol, method=method_dd.value, basis=basis_dd.value)
        notes = calc.get_educational_notes()
        if notes:
            safe = (
                notes
                .replace("**", "<b>", 1)
                .replace("**", "</b>", 1)
                .replace("\n\n", "<br><br>")
            )
            with notes_output:
                display(HTML(
                    '<div style="background:#fffbf0;padding:8px 12px;'
                    'border-radius:4px;font-size:13px;margin-top:6px">'
                    + safe + "</div>"
                ))
    except Exception:
        pass

method_dd.observe(_update_notes, names="value")
basis_dd.observe(_update_notes, names="value")


def _update_estimate(change=None):
    '''Refresh the time-estimate label based on current molecule + method/basis.'''
    mol = _state.get("molecule")
    if mol is None:
        perf_estimate_html.value = ""
        return
    try:
        est = _calc_log.estimate_time(
            n_atoms=len(mol.atoms),
            n_electrons=mol.get_electron_count(),
            method=method_dd.value,
            basis=basis_dd.value,
        )
        perf_estimate_html.value = _calc_log.format_estimate(est)
    except Exception:
        perf_estimate_html.value = ""

method_dd.observe(_update_estimate, names="value")
basis_dd.observe(_update_estimate, names="value")

# Log startup event
_calc_log.log_event("startup", f"QuantUI-local {quantui.__version__} started")


def _do_accumulate(btn):
    r = _state["last_result"]
    if r is None:
        return
    _state["results"].append(r)
    _refresh_comparison()

accumulate_btn.on_click(_do_accumulate)


def _refresh_comparison():
    from quantui import summary_from_session_result, comparison_table_html
    comparison_output.clear_output(wait=True)
    results = _state["results"]
    if not results:
        return
    summaries = [summary_from_session_result(r) for r in results]
    with comparison_output:
        display(HTML(comparison_table_html(summaries)))
        if len(summaries) > 1:
            try:
                from quantui import plot_comparison
                plot_comparison(summaries)
            except Exception:
                pass


def _do_clear(btn):
    _state["results"].clear()
    comparison_output.clear_output()

clear_btn.on_click(_do_clear)


def _do_export(btn):
    mol = _state["molecule"]
    if mol is None:
        export_status.value = "Load a molecule first."
        return
    try:
        from quantui import PySCFCalculation
        from pathlib import Path
        calc  = PySCFCalculation(mol, method=method_dd.value, basis=basis_dd.value)
        fname = f"{mol.get_formula()}_{method_dd.value}_{basis_dd.value}.py"
        calc.generate_calculation_script(Path(fname))
        export_status.value = f"Saved: {fname}"
    except Exception as exc:
        export_status.value = f"Error: {exc}"

export_btn.on_click(_do_export)


def _on_method_help(btn):
    _show_help_topic("method")

def _on_basis_help(btn):
    _show_help_topic("basis_set")

method_help_btn.on_click(_on_method_help)
basis_help_btn.on_click(_on_basis_help)


In [ ]:
# ── Preset Library ─────────────────────────────────────────────────────────
_preset_opts = ["(select a molecule)"] + list(MOLECULE_LIBRARY.keys())
preset_dd = widgets.Dropdown(
    options=_preset_opts, value="(select a molecule)",
    description="Molecule:", style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

def _load_preset(change):
    name = change["new"]
    if name.startswith("("):
        return
    d = MOLECULE_LIBRARY[name]
    _set_molecule(
        Molecule(
            atoms=d["atoms"], coordinates=d["coordinates"],
            charge=d["charge"], multiplicity=d["multiplicity"],
        ),
        d["description"],
    )

preset_dd.observe(_load_preset, names="value")

# ── XYZ Input ──────────────────────────────────────────────────────────────
xyz_area = widgets.Textarea(
    placeholder=(
        "Paste XYZ coordinates (symbol  x  y  z):\n"
        "O  0.000  0.000  0.000\n"
        "H  0.757  0.587  0.000\n"
        "H -0.757  0.587  0.000"
    ),
    layout=widgets.Layout(width="440px", height="130px"),
)
xyz_btn = widgets.Button(description="Load XYZ", button_style="info", icon="upload")
xyz_msg = widgets.Label()

def _load_xyz(btn):
    try:
        mol = parse_xyz_input(xyz_area.value.strip())
        _set_molecule(mol, "Loaded from XYZ input")
        xyz_msg.value = ""
    except Exception as exc:
        xyz_msg.value = f"Parse error: {exc}"

xyz_btn.on_click(_load_xyz)

# ── PubChem Search ─────────────────────────────────────────────────────────
pubchem_txt = widgets.Text(
    placeholder="name or SMILES  (e.g. aspirin, caffeine, CC(=O)O)",
    layout=widgets.Layout(width="380px"),
)
pubchem_btn = widgets.Button(
    description="Search", button_style="info", icon="search",
    disabled=not PUBCHEM_AVAILABLE,
    layout=widgets.Layout(width="100px"),
)
pubchem_msg = widgets.Label(
    value="" if PUBCHEM_AVAILABLE else "PubChem unavailable — check internet connection"
)

def _search_pubchem(btn):
    query = pubchem_txt.value.strip()
    if not query:
        pubchem_msg.value = "Enter a molecule name or SMILES."
        return
    if student_friendly_fetch is None:
        pubchem_msg.value = "PubChem module not available."
        return
    pubchem_msg.value = f'Searching for "{query}"...'
    pubchem_btn.disabled = True

    def _do():
        try:
            mol = student_friendly_fetch(query)
            _set_molecule(mol, f"PubChem: {query}")
            pubchem_msg.value = f"Loaded {mol.get_formula()} from PubChem."
        except Exception as exc:
            pubchem_msg.value = f"Not found: {exc}"
        finally:
            pubchem_btn.disabled = False

    threading.Thread(target=_do, daemon=True).start()

pubchem_btn.on_click(_search_pubchem)

# ── Assemble input tab ─────────────────────────────────────────────────────
_hint = '<p style="margin:4px 0 8px;color:#666;font-size:13px">'
tab_preset = widgets.VBox([
    widgets.HTML(_hint + "Choose from 20+ curated educational molecules.</p>"),
    preset_dd,
])
tab_xyz = widgets.VBox([
    widgets.HTML(_hint + "Paste XYZ coordinates (element x y z, one atom per line).</p>"),
    xyz_area,
    widgets.HBox([xyz_btn, xyz_msg]),
])
tab_pubchem = widgets.VBox([
    widgets.HTML(_hint + "Search by name or SMILES. Requires internet connection.</p>"),
    widgets.HBox([pubchem_txt, pubchem_btn]),
    pubchem_msg,
])

input_tab = widgets.Tab(children=[tab_preset, tab_xyz, tab_pubchem])
for _i, _t in enumerate(["Preset Library", "XYZ Input", "PubChem Search"]):
    input_tab.set_title(_i, _t)

# ── Collapsible molecule input container ──────────────────────────────────
mol_input_expanded = widgets.VBox([
    widgets.HTML('<h3 style="margin:8px 0 6px">Molecule Input</h3>'),
    input_tab,
])

# Change button — re-expands the input panel
change_mol_btn = widgets.Button(
    description="Change", button_style="", icon="pencil",
    layout=widgets.Layout(width="100px", height="32px"),
    tooltip="Re-expand the molecule input panel",
)

# Collapsed view: compact summary pill + Change button
mol_input_collapsed = widgets.HBox(
    [mol_summary_compact, change_mol_btn],
    layout=widgets.Layout(align_items="center", gap="12px", padding="6px 0"),
)

# Container starts expanded; _set_molecule() collapses it after first load.
mol_input_container = widgets.VBox(
    [mol_input_expanded, mol_info_html, viz_output],
    layout=widgets.Layout(margin="0 0 4px 0"),
)

def _expand_mol_input(btn):
    mol_input_container.children = [mol_input_expanded, mol_info_html, viz_output]

change_mol_btn.on_click(_expand_mol_input)


In [ ]:
calc_setup_panel = widgets.VBox([
    widgets.HTML('<h3 style="margin:14px 0 6px">Calculation Setup</h3>'),
    widgets.HBox([
        widgets.VBox([
            widgets.HBox(
                [method_dd, method_help_btn],
                layout=widgets.Layout(align_items="center", gap="4px"),
            ),
            widgets.HBox(
                [basis_dd, basis_help_btn],
                layout=widgets.Layout(align_items="center", gap="4px"),
            ),
        ]),
        widgets.HTML("&ensp;&ensp;"),
        widgets.VBox([charge_si, mult_si]),
    ]),
    calc_type_dd,
    calc_extra_opts,
    preopt_cb,
    notes_output,
])


In [ ]:
run_panel = widgets.VBox([
    widgets.HTML(
        '<h3 style="margin:14px 0 6px">Run Calculation</h3>'
        '<p style="color:#555;font-size:13px;margin:0 0 8px">PySCF runs in this kernel. '
        "Output appears live below. Large molecules or high-accuracy basis sets may take "
        "several minutes on a laptop.</p>"
    ),
    perf_estimate_html,
    widgets.HBox([run_btn, run_status]),
    widgets.HBox(
        [
            widgets.HTML(
                '<span style="font-size:13px;font-weight:600;color:#444">'
                "Calculation Output</span>"
            ),
            log_clear_btn,
        ],
        layout=widgets.Layout(align_items="center", justify_content="space-between",
                              margin="10px 0 4px", max_width="460px"),
    ),
    run_output,
])


In [ ]:
results_panel = widgets.VBox([
    widgets.HTML('<h3 style="margin:14px 0 6px">Results</h3>'),
    result_output,
])


In [ ]:
import json
from pathlib import Path

# ── Past Results browser ──────────────────────────────────────────────────────
past_dd = widgets.Dropdown(
    description="Load:",
    options=[("(no saved results)", "")],
    style={"description_width": "50px"},
    layout=widgets.Layout(width="500px"),
)
past_refresh_btn = widgets.Button(
    description="Refresh", button_style="", icon="refresh",
    layout=widgets.Layout(width="100px"),
    tooltip="Rescan the results directory",
)
copy_path_btn = widgets.Button(
    description="Copy path", button_style="", icon="clipboard",
    layout=widgets.Layout(width="120px"),
    tooltip="Copy the results directory path to clipboard",
)
results_path_lbl = widgets.HTML()
past_output = widgets.Output()


def _get_results_dir() -> Path:
    from quantui.results_storage import _default_results_dir
    return _default_results_dir().resolve()


def _format_past_result(data: dict) -> str:
    _conv = "Yes" if data.get("converged") else "No (treat results with caution)"
    _cc   = "green" if data.get("converged") else "#c00"
    _gap  = (
        f"{data['homo_lumo_gap_ev']:.4f} eV"
        if data.get("homo_lumo_gap_ev") is not None else "N/A"
    )
    _rows = "".join(
        f'<tr>'
        f'<td style="padding:3px 18px 3px 0;color:#444">{k}</td>'
        f'<td style="color:{vc}">{v}</td>'
        f"</tr>"
        for k, v, vc in [
            ("Total energy",
             f"{data['energy_hartree']:.8f} Ha &ensp;({data['energy_ev']:.4f} eV)", "#000"),
            ("HOMO-LUMO gap",   _gap, "#000"),
            ("SCF converged",   _conv, _cc),
            ("SCF iterations",  str(data.get("n_iterations", "?")), "#000"),
        ]
    )
    ts = data.get("timestamp", "")
    return (
        f'<div style="background:#f0fff0;border-left:4px solid #4CAF50;'
        f'padding:10px 14px;border-radius:4px;margin:6px 0">'
        f'<b>{data["formula"]} &mdash; {data["method"]}/{data["basis"]}</b>'
        f'&ensp;<small style="color:#777">{ts}</small>'
        f'<table style="margin-top:8px;font-size:14px;border-collapse:collapse">'
        f"{_rows}</table></div>"
    )


def _refresh_results_browser():
    '''Repopulate the past-results dropdown. Called on startup and after each run.'''
    try:
        from quantui import list_results, load_result
    except ImportError:
        return
    results_path_lbl.value = f'<span style="font-size:13px;color:#64748b">{_get_results_dir()}</span>'
    dirs = list_results()
    if not dirs:
        past_dd.options = [("(no saved results)", "")]
        return
    options = []
    for d in dirs:
        try:
            data = load_result(d)
            ts    = data.get("timestamp", d.name)
            label = f"{ts}  ·  {data['formula']}  {data['method']}/{data['basis']}"
            options.append((label, str(d)))
        except Exception:
            pass
    past_dd.options = options if options else [("(no saved results)", "")]


def _load_past_result(change):
    path_str = change["new"]
    if not path_str:
        past_output.clear_output()
        return
    past_output.clear_output(wait=True)
    try:
        from quantui import load_result
        data = load_result(Path(path_str))
        past_output.append_display_data(HTML(_format_past_result(data)))
    except Exception as exc:
        past_output.append_stdout(f"Could not load result: {exc}\n")


def _copy_results_path(btn):
    '''Copy the results directory path to clipboard via browser JS.'''
    p = _get_results_dir()
    p.mkdir(parents=True, exist_ok=True)
    path_str = str(p).replace("\\", "\\\\").replace("'", "\\'")
    from IPython.display import Javascript
    display(Javascript(f"navigator.clipboard.writeText(\'{path_str}\')"))
    # Brief confirmation in the label
    import threading
    results_path_lbl.value = (
        f'<span style="color:#22c55e;font-size:13px">Copied: {p}</span>'
    )
    def _reset():
        import time; time.sleep(3)
        results_path_lbl.value = f'<span style="font-size:13px;color:#64748b">{p}</span>'
    threading.Thread(target=_reset, daemon=True).start()


past_dd.observe(_load_past_result, names="value")
past_refresh_btn.on_click(lambda _: _refresh_results_browser())
copy_path_btn.on_click(_copy_results_path)

# Populate on startup and make the real function visible to _do_run in Cell 3.
_refresh_results_browser()

# ── Performance stats widgets ───────────────────────────────────────────────────────────────────
_perf_stats_html = widgets.HTML()
_perf_events_html = widgets.HTML()

_reset_btn = widgets.Button(
    description="Reset performance database",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width="230px"),
)
_reset_confirm_html = widgets.HTML(
    '<span style="color:#dc2626;font-size:13px">'
    "<b>Warning:</b> This will permanently delete all performance records. "
    "Time estimates will reset to &ldquo;no data&rdquo;.</span>"
)
_reset_confirm_yes = widgets.Button(
    description="Yes, delete all records",
    button_style="danger",
    icon="check",
    layout=widgets.Layout(width="190px"),
)
_reset_confirm_no = widgets.Button(
    description="Cancel",
    button_style="",
    icon="times",
    layout=widgets.Layout(width="90px"),
)
_reset_confirm_box = widgets.VBox(
    [
        _reset_confirm_html,
        widgets.HBox(
            [_reset_confirm_yes, _reset_confirm_no],
            layout=widgets.Layout(gap="8px", margin="4px 0 0"),
        ),
    ],
    layout=widgets.Layout(
        display="none",
        border="1px solid #fca5a5",
        padding="8px 10px",
        margin="6px 0 0",
    ),
)


def _build_perf_stats_html() -> str:
    from quantui.calc_log import get_perf_history
    records = get_perf_history()
    if not records:
        return (
            '<span style="color:#94a3b8;font-size:13px">'
            "No performance data recorded yet.</span>"
        )
    groups: dict = {}
    for r in records:
        key = (r.get("method", "?"), r.get("basis", "?"))
        groups.setdefault(key, []).append(r)
    rows = ""
    for (meth, bas), recs in sorted(groups.items()):
        times = [r["elapsed_s"] for r in recs if "elapsed_s" in r]
        n = len(recs)
        if times:
            avg = sum(times) / len(times)
            rows += (
                "<tr>"
                f'<td style="padding:2px 12px 2px 0">{meth}</td>'
                f'<td style="padding:2px 12px 2px 0">{bas}</td>'
                f'<td style="padding:2px 12px 2px 0;text-align:right">{n}</td>'
                f'<td style="padding:2px 12px 2px 0;text-align:right">{avg:.1f} s</td>'
                f'<td style="padding:2px 12px 2px 0;text-align:right">{min(times):.1f} s</td>'
                f'<td style="padding:2px 12px 2px 0;text-align:right">{max(times):.1f} s</td>'
                "</tr>"
            )
    header = (
        "<tr>"
        '<th style="text-align:left;padding:2px 12px 2px 0;color:#64748b">Method</th>'
        '<th style="text-align:left;padding:2px 12px 2px 0;color:#64748b">Basis</th>'
        '<th style="text-align:right;padding:2px 12px 2px 0;color:#64748b">Runs</th>'
        '<th style="text-align:right;padding:2px 12px 2px 0;color:#64748b">Avg</th>'
        '<th style="text-align:right;padding:2px 12px 2px 0;color:#64748b">Min</th>'
        '<th style="text-align:right;padding:2px 12px 2px 0;color:#64748b">Max</th>'
        "</tr>"
    )
    return (
        '<table style="font-size:13px;border-collapse:collapse;width:100%">'
        f"{header}{rows}</table>"
    )


def _build_events_html() -> str:
    from quantui.calc_log import get_recent_events
    events = get_recent_events(20)
    if not events:
        return (
            '<span style="color:#94a3b8;font-size:13px">'
            "No events recorded yet.</span>"
        )
    rows = ""
    for e in reversed(events):
        ts = e.get("timestamp", "")[:19].replace("T", " ")
        evt = e.get("event", "")
        msg = e.get("message", "")
        rows += (
            "<tr>"
            f'<td style="padding:1px 10px 1px 0;color:#94a3b8;font-size:11px;white-space:nowrap">{ts}</td>'
            f'<td style="padding:1px 10px 1px 0;color:#475569;font-size:12px">{evt}</td>'
            f'<td style="padding:1px 0;color:#334155;font-size:12px">{msg}</td>'
            "</tr>"
        )
    return (
        '<table style="font-size:13px;border-collapse:collapse;width:100%">'
        f"{rows}</table>"
    )


def _refresh_perf_stats():
    _perf_stats_html.value = _build_perf_stats_html()
    _perf_events_html.value = _build_events_html()


def _on_reset_click(btn):
    _reset_confirm_box.layout.display = ""


def _on_confirm_yes(btn):
    from quantui.calc_log import reset_perf_log
    reset_perf_log()
    _reset_confirm_box.layout.display = "none"
    _refresh_perf_stats()


def _on_confirm_no(btn):
    _reset_confirm_box.layout.display = "none"


_reset_btn.on_click(_on_reset_click)
_reset_confirm_yes.on_click(_on_confirm_yes)
_reset_confirm_no.on_click(_on_confirm_no)
_refresh_perf_stats()

_perf_stats_panel = widgets.VBox([
    _perf_stats_html,
    widgets.HTML(
        '<p style="margin:10px 0 4px;color:#475569;font-size:13px;font-weight:600">'
        "Recent events (last 20)</p>"
    ),
    _perf_events_html,
    widgets.HBox(
        [_reset_btn],
        layout=widgets.Layout(margin="14px 0 4px"),
    ),
    _reset_confirm_box,
])

_perf_accordion = widgets.Accordion(children=[_perf_stats_panel], selected_index=None)
_perf_accordion.set_title(0, "Performance stats")

# ── History panel (assembled widget for the History tab) ─────────────────
history_panel = widgets.VBox([
    widgets.HTML(
        '<p style="color:#555;font-size:13px;margin:0 0 8px">'
        "Calculations are saved automatically. Select one below to view its results.</p>"
    ),
    widgets.HBox(
        [past_dd, past_refresh_btn, copy_path_btn],
        layout=widgets.Layout(align_items="center", gap="8px"),
    ),
    results_path_lbl,
    past_output,
    _perf_accordion,
])


In [ ]:
from pathlib import Path

# ── Compare tab widgets ────────────────────────────────────────────────────────
compare_select = widgets.SelectMultiple(
    options=[("(no saved results)", "")],
    rows=8,
    description="",
    layout=widgets.Layout(width="100%"),
)
compare_refresh_btn = widgets.Button(
    description="Refresh", button_style="", icon="refresh",
    layout=widgets.Layout(width="100px"),
)
compare_btn = widgets.Button(
    description="Compare selected", button_style="primary", icon="bar-chart",
    disabled=True,
    layout=widgets.Layout(width="180px"),
)
compare_clear_btn = widgets.Button(
    description="Clear", button_style="warning", icon="times",
    layout=widgets.Layout(width="90px"),
)
compare_output = widgets.Output()


def _populate_compare_list():
    '''Repopulate compare_select with all saved results.'''
    from quantui.results_storage import list_results, load_result
    dirs = list_results()
    if not dirs:
        compare_select.options = [("(no saved results)", "")]
        compare_btn.disabled = True
        return
    options = []
    for d in dirs:
        try:
            data = load_result(d)
            ts = data.get("timestamp", d.name[:19])
            label = f"{ts}  {data['formula']}  {data['method']}/{data['basis']}"
            options.append((label, str(d)))
        except Exception:
            options.append((d.name, str(d)))
    compare_select.options = options
    compare_btn.disabled = False


def _on_compare_refresh(btn):
    _populate_compare_list()


def _on_compare(btn):
    selected = compare_select.value  # tuple of selected path strings
    if not selected or selected == ("",):
        return
    compare_output.clear_output(wait=True)
    from quantui.results_storage import load_result
    from quantui import summary_from_saved_result, comparison_table_html, plot_comparison
    summaries = []
    for path_str in selected:
        if not path_str:
            continue
        try:
            data = load_result(Path(path_str))
            summaries.append(summary_from_saved_result(data))
        except Exception as exc:
            with compare_output:
                display(HTML(f'<p style="color:#ef4444">Error loading result: {exc}</p>'))
    if not summaries:
        return
    with compare_output:
        display(HTML(comparison_table_html(summaries)))
        if len(summaries) > 1:
            try:
                import matplotlib.pyplot as plt
                fig = plot_comparison(summaries)
                display(fig)
                plt.close(fig)
            except Exception:
                pass


def _on_compare_clear(btn):
    compare_select.value = ()
    compare_output.clear_output()


compare_refresh_btn.on_click(_on_compare_refresh)
compare_btn.on_click(_on_compare)
compare_clear_btn.on_click(_on_compare_clear)

# Populate on load; expose to Cell-3 placeholder so _do_run refreshes it.
_populate_compare_list()
globals()["_populate_compare_list"] = _populate_compare_list

# ── Compare panel (assembled widget for the Compare tab) ──────────────────────
compare_panel = widgets.VBox([
    widgets.HTML(
        '<h3 style="margin:14px 0 6px">Compare Calculations</h3>'
        '<p style="color:#555;font-size:13px;margin:0 0 8px">'
        "Select two or more saved calculations to compare side-by-side. "
        "Hold Ctrl (or ⌘) to select multiple entries.</p>"
    ),
    widgets.HBox([compare_refresh_btn]),
    compare_select,
    widgets.HBox(
        [compare_btn, compare_clear_btn],
        layout=widgets.Layout(gap="8px", margin="6px 0"),
    ),
    compare_output,
], layout=widgets.Layout(padding="8px 0"))

# ── Advanced accordion (Export only) ──────────────────────────────────────────
_export_content = widgets.VBox([
    widgets.HTML(
        '<p style="color:#555;font-size:13px;margin:0 0 8px">'
        "Download a self-contained PySCF script you can study or run outside the notebook.</p>"
    ),
    widgets.HBox([export_btn, export_status]),
])
advanced_accordion = widgets.Accordion(children=[_export_content])
advanced_accordion.set_title(0, "Export Script")
advanced_accordion.selected_index = None  # collapsed by default


In [ ]:
from quantui.help_content import HELP_TOPICS

# ── Help tab content ──────────────────────────────────────────────────────────
_help_keys   = list(HELP_TOPICS.keys())
_help_labels = [HELP_TOPICS[k]["title"] for k in _help_keys]

help_topic_dd = widgets.Dropdown(
    options=list(zip(_help_labels, _help_keys)),
    description="Topic:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="460px"),
)
help_content_html = widgets.HTML()

def _render_help_topic(change=None):
    key = help_topic_dd.value
    if key and key in HELP_TOPICS:
        entry = HELP_TOPICS[key]
        help_content_html.value = (
            f'<div style="border:1px solid #e2e8f0;border-radius:6px;'
            f'padding:14px 18px;margin:8px 0;background:#f8fafc;max-width:700px">'
            f'<h4 style="margin:0 0 10px;color:#1e293b;font-size:15px;font-weight:700">'
            f'{entry["title"]}</h4>'
            f'<div style="font-size:14px;color:#334155;line-height:1.6">'
            f'{entry["body"]}</div>'
            f'</div>'
        )

help_topic_dd.observe(_render_help_topic, names="value")
_render_help_topic()  # render first topic on load

help_tab_panel = widgets.VBox([
    widgets.HTML(
        '<p style="color:#555;font-size:13px;margin:4px 0 12px">'
        "Browse help topics below. Click <b>?</b> next to the Method or Basis Set "
        "dropdown in the Calculate tab to jump directly to a relevant topic.</p>"
    ),
    help_topic_dd,
    help_content_html,
], layout=widgets.Layout(padding="8px 0"))

# ── Assemble root tab ─────────────────────────────────────────────────────────
_calculate_content = widgets.VBox([
    step_progress.widget,
    mol_input_container,
    calc_setup_panel,
    run_panel,
    results_panel,
    advanced_accordion,
], layout=widgets.Layout(padding="8px 0"))

root_tab = widgets.Tab(children=[
    _calculate_content,
    history_panel,
    compare_panel,
    help_tab_panel,
])
root_tab.set_title(0, "Calculate")
root_tab.set_title(1, "History")
root_tab.set_title(2, "Compare")
root_tab.set_title(3, "Help")

# ── "?" button callbacks: jump to Help tab with specific topic ────────────────
def _show_help_topic(topic: str) -> None:
    if topic in HELP_TOPICS:
        help_topic_dd.value = topic
    root_tab.selected_index = 3

# Overwrite Cell-3 placeholder so "?" buttons find the real function.
globals()["_show_help_topic"] = _show_help_topic

# ── Single root display call ──────────────────────────────────────────────────
display(root_tab)
